# 01 — Data audit

First-stage sanity check on the raw inputs that flow into the tokenisation pipeline.
Run this notebook after `rt-fetch --site both`.

**What we verify here:**
1. Crash CSVs are present and parseable
2. Severity / year distributions look sensible (no large gaps)
3. Coordinate ranges fall inside the expected country envelopes
4. The SHA-256 manifest at `data/raw/manifest.json` agrees with what's on disk
5. A small sample OSM extract for Cambridge actually fetches

If anything below errors out, the downstream pipeline cannot succeed — fix at this stage rather than chasing symptoms in `tokenise.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
print('project root:', ROOT)
print('raw data:   ', RAW)

## Manifest sanity

In [ ]:
manifest = json.loads((RAW / 'manifest.json').read_text())

rows = []
for relpath, meta in manifest['files'].items():
    p = RAW / relpath
    actual_size = p.stat().st_size if p.exists() else None
    rows.append({
        'file': relpath,
        'on_disk_mb': round(actual_size / 1e6, 1) if actual_size else None,
        'manifest_mb': round(meta['size_bytes'] / 1e6, 1),
        'size_match': actual_size == meta['size_bytes'],
        'licence': meta['licence'][:60] + '...' if len(meta['licence']) > 60 else meta['licence'],
    })
pd.DataFrame(rows)

## UK STATS19 collisions

In [ ]:
uk = pd.read_csv(
    RAW / 'uk' / 'stats19_collision_last_5_years.csv',
    usecols=['collision_year', 'collision_severity', 'longitude', 'latitude', 'urban_or_rural_area'],
    low_memory=False,
)
print(f'rows: {len(uk):,}')
print()
print('collisions per year:')
print(uk.collision_year.value_counts().sort_index())
print()
print('severity (1=fatal, 2=serious, 3=slight):')
print(uk.collision_severity.value_counts().sort_index())
print()
print('urban/rural (1=urban, 2=rural):')
print(uk.urban_or_rural_area.value_counts().sort_index())
print()
print('lon/lat envelope:')
print(uk[['longitude', 'latitude']].agg(['min', 'max', 'mean']).round(3))

Sanity checks:
- ~5 years of data, all years roughly equal in count
- severity skewed toward `slight` (most crashes), as expected
- lon/lat envelope: GB ≈ longitude in `[-8, 2]`, latitude in `[49.8, 60.9]`

## NZ CAS

In [ ]:
nz = pd.read_csv(
    RAW / 'nz' / 'CAS_Data_public.csv',
    usecols=['crashYear', 'crashSeverity', 'X', 'Y', 'fatalCount', 'seriousInjuryCount', 'minorInjuryCount'],
    low_memory=False,
)
print(f'rows: {len(nz):,}')
print()
print('crashes per year (last 12 yrs):')
print(nz[nz.crashYear >= nz.crashYear.max() - 11].crashYear.value_counts().sort_index())
print()
print('severity:')
print(nz.crashSeverity.value_counts())
print()
print('X/Y envelope (should be WGS84 lon/lat since we requested spatialRefId=4326):')
print(nz[['X', 'Y']].agg(['min', 'max', 'mean']).round(3))
print()
print('total casualties by severity:')
print(nz[['fatalCount', 'seriousInjuryCount', 'minorInjuryCount']].sum())

Sanity checks:
- NZ ≈ longitude in `[166, 179]`, latitude in `[-47, -34]`
- `crashSeverity` is text: Non-Injury / Minor / Serious / Fatal
- annual counts should be in the 20–30 k range

## OSM extract sanity (small Cambridge bbox)

In [ ]:
import osmnx as ox

BBOX_CAM = (0.115, 52.200, 0.135, 52.215)  # small central Cambridge
G = ox.graph_from_bbox(BBOX_CAM, network_type='drive', simplify=True)
edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
print(f'edges: {len(edges)}')
print()
print('highway class counts:')
print(edges.highway.apply(lambda x: x[0] if isinstance(x, list) else x).value_counts())
print()
print('share with explicit maxspeed:')
have_speed = edges.maxspeed.notna().mean()
print(f'{have_speed:.1%}')

## Tokeniser smoke (does the full pipeline run?)

In [ ]:
from road_tokeniser.tokenise import run

OUT = ROOT / 'webapp' / 'tokens_audit_cambridge_small.geojson'
g = run(BBOX_CAM, site='uk', token_len=25.0, out=OUT)
print(f'wrote {len(g)} tokens, columns: {list(g.columns)[:8]}...')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flat

g['length_m'].hist(bins=40, ax=next(axes), color='#2b5d8b')
axes[0].set_title('Token length (m) — target 25'); axes[0].axvline(25, color='red', ls='--', lw=1)

g['posted_speed_kph'].value_counts().sort_index().plot.bar(ax=next(axes), color='#2b5d8b')
axes[1].set_title('Posted speed (kph)'); axes[1].tick_params(axis='x', rotation=0)

g['safe_system_speed_kph'].value_counts().sort_index().plot.bar(ax=next(axes), color='#1a9850')
axes[2].set_title('Safe System recommended (kph)'); axes[2].tick_params(axis='x', rotation=0)

g['misalignment_kph'].hist(bins=20, ax=next(axes), color='#d73027')
axes[3].set_title('Misalignment (kph)')

g['vru_score'].hist(bins=20, ax=next(axes), color='#9e3a8a')
axes[4].set_title('VRU exposure score')

g['priority_score'].hist(bins=20, ax=next(axes), color='#d97706')
axes[5].set_title('Priority score')

for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()

## Rule firing breakdown

In [ ]:
rules_fired = g.groupby('safe_system_rule').agg(
    n_tokens=('token_id', 'size'),
    pct=('token_id', lambda s: f'{100*len(s)/len(g):.1f}%'),
    avg_misalignment=('misalignment_kph', 'mean'),
    max_priority=('priority_score', 'max'),
).round(2).sort_values('n_tokens', ascending=False)
rules_fired

## Top 20 priority segments — eyeball check

These are the segments the rule engine considers highest priority for review. Skim the list:
- Do the street names look like real corridors with VRU exposure?
- Is the `safe_system_rule` consistent with the posted/recommended pair?
- Are any obvious OSM tagging errors (e.g. residential street tagged at 80 km/h) surfacing?

In [ ]:
g.nlargest(20, 'priority_score')[
    ['name', 'highway', 'posted_speed_kph', 'safe_system_speed_kph',
     'misalignment_kph', 'safe_system_rule', 'vru_score', 'crash_count']
]